### POPISNÉ MODELY PRE ANALÝZU DÁT O PACIENTOCH S COVID-19

##### Všetky vlny spolu
Vizualizácia rozloženia cieľového atribútu a chýbajúcich hodnôt atribútov

Autor: Bc. Šimon Mikolaj

In [ ]:
import pandas as pd
import numpy as np
import re

# Načítanie Excelového súboru
file_path = "Vsetky_vlny_spolu.xlsx"
df = pd.read_excel(file_path)

# Identifikácia skupín atribútov laboratórnych testov
pattern = re.compile(r'(.+)\s+(first|last|min|max)$', re.IGNORECASE)
grouped = {}

for col in df.columns:
    match = pattern.match(str(col).strip())
    if match:
        base_name = match.group(1).strip()
        suffix = match.group(2).lower()
        grouped.setdefault(base_name, []).append(suffix)

# Určenie, ktoré sú laboratórne skupiny - first,last,min,max
bio_groups = {base for base, suffixes in grouped.items() if len(suffixes) > 1}

# Výber stĺpcov 
cols_to_keep = []
for col in df.columns:
    match = pattern.match(str(col).strip())
    if match:
        base_name = match.group(1).strip()
        suffix = match.group(2).lower()
        # Ak ide o laboratórny test, nechaj len 'last'
        if base_name in bio_groups and suffix == 'last':
            cols_to_keep.append(col)
    else:
        # Nejde o laboratórny atribút → nechaj ho
        cols_to_keep.append(col)

df = df[cols_to_keep]

# Odstránenie všetkých stĺpcov s veľkým počtom chýbajúcich hodnôt - podľa analýzy literatúry nad 25% a takých, ktoré nemajú žiadne chýbajúce hodnoty
remove_prefixes = ["Typ vakcíny", "S-IgA", "S-Chol", "S-Ig M", "S-IgG", "S-AMS", "S-LD", "S-KM", "Pohlavie", "Vek", "Fajčenie", "Alkohol", "Hypertenzia", "Diabetes mellitus", "Kardiovaskulárne ochorenia", "Chronické respiračné ochorenia", "Renálne ochorenia", "Pečeňové ochorenia", "Onkologické ochorenia", "Imunosupresia", "Vakcinácia", "Počet dávok", "Prekonal COVID-19", "Dátum príjmu", "Kód príjmu", "Meno"]
cols_to_drop = [
    col for col in df.columns 
    if any(col.strip().lower().startswith(prefix.lower()) for prefix in remove_prefixes)
]
df = df.drop(columns=cols_to_drop, errors='ignore')

# Odstránenie atribútu "Poradie"
if "Poradie" in df.columns:
    df = df.drop(columns=["Poradie"])

# Odstránenie atribútov od 1. po 18. stĺpec (vrátane) - atributy s klinickým popisom
if len(df.columns) >= 22:
    cols_to_drop = df.columns[1:18]  
    df = df.drop(columns=cols_to_drop)

df = df.dropna(axis=1, how='all')                   # Odstránenie prázdnych stĺpcov 
df = df.replace({True: 1, False: 0})                # TRUE/FALSE → 1/0 

# Spracovanie atribútu "Eo abs"
if "Eo abs last" in df.columns:
    # Najprv prekonvertuj stĺpec na čísla (ak tam boli texty, premenia sa na NaN)
    df["Eo abs last"] = pd.to_numeric(df["Eo abs last"], errors='coerce')
    # Potom nahraď 0 za NaN
    df["Eo abs last"] = df["Eo abs last"].replace(0, np.nan)


# Výpočet chýbajúcich hodnôt 
missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100
missing_table = pd.DataFrame({
    'Chýbajúce hodnoty': missing_counts,
    'Chýbajúce hodnoty (%)':  missing_counts.astype(str) + " (" +  missing_percent.round(2).astype(str) + " %)"
}).sort_values(by='Chýbajúce hodnoty', ascending=False)

# Výstup 
print("\nPrehľad chýbajúcich hodnôt:\n")
print(missing_table)

In [ ]:
# Absolútny a relatívny počet chýbajúcich atribútov
def missing_overall(df):
    total_missing = df.isna().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    percent = (total_missing / total_cells) * 100
    print("Celkový počet chýbajúcich hodnôt:", total_missing)
    print(f"Percento chýbajúcich hodnôt v celom DataFrame: {percent:.2f}%")
    return total_missing, percent

missing_overall(df)

In [ ]:
# Rozloženie závažnosti priebehu ochorenia

import matplotlib.pyplot as plt
mapping = {
    1.0: "1",
    2.0: "2",
    3.0: "3"
}

counts = (
    df["Závažnosť priebehu ochorenia"]
    .map(mapping)
    .value_counts()
    .sort_index()
)
# Výpočet percent
percentages = (counts / counts.sum()) * 100

# vizualizácia
plt.figure(figsize=(6, 5))
ax = counts.plot(kind="bar")
plt.xlabel("Závažnosť priebehu ochorenia")
plt.ylabel("Počet záznamov")
plt.xticks(rotation=360)

# Pridanie popisov (počet + percento) nad stĺpce
for i, (count, percent) in enumerate(zip(counts, percentages)):
    ax.text(
        i,
        count,
        f"{count} ({percent:.1f}%)",
        ha="center",
        va="bottom"
    )

plt.show()

In [ ]:
#  Odstránenie liekových atribútov od "MD652 | FABIFLU TABLETS" po "ANOPYRIN" 
start_col = "MD652 | FABIFLU TABLETS"
end_col = "ANOPYRIN"

if start_col in df.columns and end_col in df.columns:
    start_idx = df.columns.get_loc(start_col)
    end_idx = df.columns.get_loc(end_col)
    if start_idx <= end_idx:
        cols_to_drop = df.columns[start_idx:end_idx + 1]
        df = df.drop(columns=cols_to_drop)

In [ ]:
### zistenie typu chybajucich hodnot
import seaborn as sns
import matplotlib.pyplot as plt

# Maska chýbajúcich hodnôt
missing_mask = df.isna()

# Korelačná matica medzi maskou chýbajúcich hodnôt
corr_missing = missing_mask.corr()

plt.figure(figsize=(15,12))
sns.heatmap(corr_missing, cmap="coolwarm", center=0)
plt.title("Korelácie medzi chýbajúcimi hodnotami atribútov")
plt.show()